# BioPortal: Thyroid Cancer Ontology
Explore the Thyroid Cancer Ontology (TCO) via the BioPortal REST API to understand its schema and find genomics-related terms.

## 1. Set Up Environment and Config
Import libraries and define the base URL, timeout, and helper utilities.

In [3]:
import os
import time
import requests
import pandas as pd
from typing import Any, Dict, Iterable, List, Optional

BASE_URL = "https://data.bioontology.org"
TIMEOUT = 30

session = requests.Session()


def _sleep_if_rate_limited(response: requests.Response, delay_seconds: float = 1.0) -> None:
    if response.status_code == 429:
        time.sleep(delay_seconds)


def api_get(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    url = f"{BASE_URL}{endpoint}"
    response = session.get(url, params=params or {}, timeout=TIMEOUT)
    _sleep_if_rate_limited(response)
    response.raise_for_status()
    return response.json()

/Users/cindyz1/bime/550/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Configure BioPortal API Access
Load the API key from environment variables and attach it to requests.

In [4]:
# get BIOPORTAL_API_KEY from dotenv
API_KEY = os.getenv("BIOPORTAL_API_KEY")


def api_get_auth(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    params = dict(params or {})
    if API_KEY:
        params.setdefault("apikey", API_KEY)
    return api_get(endpoint, params=params)


if not API_KEY:
    print("⚠️ Set BIOPORTAL_API_KEY in your environment to avoid limited access.")

## 3. Search and Identify Thyroid Cancer Ontology
Search BioPortal and select the most relevant ontology entry.

In [5]:
search_results = api_get_auth("/search", params={"q": "Thyroid Cancer Ontology"})
collection = search_results.get("collection", [])

search_df = pd.DataFrame([
    {
        "name": item.get("prefLabel") or item.get("name"),
        "acronym": item.get("acronym"),
        "id": item.get("@id"),
        "description": (item.get("definition") or ["-"])[0] if isinstance(item.get("definition"), list) else item.get("definition"),
        "ontology_id": item.get("ontologyId") or item.get("ontology"),
    }
    for item in collection
])

search_df.head(10)

,name,acronym,id,description,ontology_id
0,Thyroid cancer,None,http://purl.bioontology.org/ontology/MEDDRA/10...,None,None
1,Thyroid cancer,None,http://purl.bioontology.org/ontology/LNC/LA156...,None,None
2,Thyroid cancer,None,http://purl.bioontology.org/ontology/OMIM/MTHU...,None,None
3,Thyroid Cancer,None,http://purl.bioontology.org/ontology/MEDLINEPL...,<h3>What is thyroid cancer?</h3> <p>Thyroid ca...,None
4,thyroid cancer,None,http://purl.bioontology.org/ontology/PDQ/CDR00...,None,None
5,thyroid cancer,None,http://purl.obolibrary.org/obo/MONDO_0002108,A malignant neoplasm involving the thyroid gland,None
6,thyroid cancer,None,http://purl.obolibrary.org/obo/DOID_1781,An endocrine gland cancer located in the thyro...,None
7,thyroid cancer,None,http://sbmi.uth.tmc.edu/ontology/ochv#2424,None,None
8,thyroid cancer,None,http://sbmi.uth.tmc.edu/ontology/ochv#C1318516,None,None
9,thyroid cancer,None,http://sbmi.uth.tmc.edu/ontology/ochv#C0007115,None,None


## 4. Fetch Ontology Metadata and Schema Summary
Retrieve ontology metadata to understand the schema and available properties.

In [13]:
TCO_ACRONYM = "TCO"
ontology_meta = api_get_auth(f"/ontologies/{TCO_ACRONYM}")
latest_submission = api_get_auth(f"/ontologies/{TCO_ACRONYM}/latest_submission")

summary = {
    "acronym": ontology_meta.get("acronym"),
    "name": ontology_meta.get("name"),
    "description": latest_submission.get("description"),
    "version": latest_submission.get("version"),
    "released": latest_submission.get("released"),
    "status": latest_submission.get("status"),
    "homepage": latest_submission.get("homepage"),
    "contact": latest_submission.get("contact") or latest_submission.get("contacts"),
    "hasOntologyLanguage": latest_submission.get("hasOntologyLanguage"),
    "identifier": latest_submission.get("id"),
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

,value
acronym,TCO
name,Thyroid Cancer Ontology
description,Thyroid cancer ontology (TCO) contains 578 con...
version,1.0
released,2020-07-02T00:00:00.000+00:00
status,beta
homepage,None
contact,[{'id': 'https://data.bioontology.org/contacts...
hasOntologyLanguage,OWL
identifier,None


## 5. Explore Classes and Properties
List top-level classes and sample class pages to inspect schema structure.

In [14]:
def normalize_class(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "prefLabel": item.get("prefLabel") or item.get("label"),
        "@id": item.get("@id"),
        "@type": item.get("@type"),
        "ontology": item.get("links", {}).get("ontology"),
        "children": item.get("links", {}).get("children"),
    }


roots = api_get_auth(f"/ontologies/{TCO_ACRONYM}/classes/roots")
root_df = pd.DataFrame([normalize_class(item) for item in roots.get("collection", [])])
root_df.head(20)

AttributeError: 'list' object has no attribute 'get'

In [ ]:
def list_classes(page: int = 1, pagesize: int = 25) -> pd.DataFrame:
    classes = api_get_auth(
        f"/ontologies/{TCO_ACRONYM}/classes",
        params={"page": page, "pagesize": pagesize}
    )
    return pd.DataFrame([normalize_class(item) for item in classes.get("collection", [])])


classes_page_1 = list_classes(page=1, pagesize=25)
classes_page_1.head(10)

In [ ]:
try:
    properties = api_get_auth(f"/ontologies/{TCO_ACRONYM}/properties")
    properties_df = pd.DataFrame([
        {
            "prefLabel": item.get("prefLabel") or item.get("label"),
            "@id": item.get("@id"),
            "@type": item.get("@type"),
        }
        for item in properties.get("collection", [])
    ])
    properties_df.head(20)
except requests.HTTPError as exc:
    print(f"Properties endpoint not available: {exc}")

## 6. Query for Genomics-Related Terms
Search within the ontology for genomics keywords (gene, mutation, variant, etc.).

In [9]:
genomics_keywords = ["gene", "mutation", "variant", "genomic", "dna", "rna", "fusion", "copy number", "expression"]

results: List[Dict[str, Any]] = []
for keyword in genomics_keywords:
    data = api_get_auth(
        "/search",
        params={"q": keyword, "ontologies": TCO_ACRONYM}
    )
    for item in data.get("collection", []):
        results.append({
            "query": keyword,
            "prefLabel": item.get("prefLabel") or item.get("label"),
            "@id": item.get("@id"),
            "@type": item.get("@type"),
            "definition": (item.get("definition") or [""])[0] if isinstance(item.get("definition"), list) else item.get("definition"),
        })

results_df = pd.DataFrame(results).drop_duplicates(subset=["@id"]).reset_index(drop=True)
results_df.head(20)

,query,prefLabel,@id,@type,definition
0,gene,Gene,http://www.projecthalo.com/aura#Gene,http://www.w3.org/2002/07/owl#Class,None
1,gene,Gene,http://www.case.edu/EpilepsyOntology.owl#Gene,http://www.w3.org/2002/07/owl#Class,None
2,gene,Gene,http://www.co-ode.org/ontologies/galen#Gene,http://www.w3.org/2002/07/owl#Class,None
3,gene,Gene,http://www.semanticweb.org/rjyy/ontologies/201...,http://www.w3.org/2002/07/owl#Class,None
4,gene,Gene,http://purl.org/skeletome/bonedysplasia#Gene,http://www.w3.org/2002/07/owl#Class,A functional unit of heredity which occupies a...
5,gene,gene,http://purl.org/biotop/biotop.owl#Gene,http://www.w3.org/2002/07/owl#Class,None
6,gene,Gene,http://www.biopax.org/release/biopax-level3.ow...,http://www.w3.org/2002/07/owl#Class,Definition: A continuant that encodes informat...
7,gene,Gene,http://www.imgt.org/download/IMGT-ONTOLOGY/IMG...,http://www.w3.org/2002/07/owl#Class,"The ""Gene"" concept allows to classify a unit o..."
8,gene,gene,http://www.imgt.org/download/IMGT-ONTOLOGY/IMG...,http://www.w3.org/2002/07/owl#Class,Identifies a gDNA sequence unit that can be po...
9,gene,Gene,http://www.cvrgrid.org/ontologies/Electrophysi...,http://www.w3.org/2002/07/owl#Class,None


## 7. Parse and Normalize Results
Flatten nested fields into tabular data and standardize identifiers.

In [10]:
def normalize_text_list(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return "; ".join(str(v) for v in value if v)
    return str(value)


normalized_results = results_df.copy()
normalized_results["prefLabel"] = normalized_results["prefLabel"].fillna("")
normalized_results["definition"] = normalized_results["definition"].apply(normalize_text_list)
normalized_results["short_id"] = normalized_results["@id"].str.split("/").str[-1]

normalized_results.head(20)

,query,prefLabel,@id,@type,definition,short_id
0,gene,Gene,http://www.projecthalo.com/aura#Gene,http://www.w3.org/2002/07/owl#Class,,aura#Gene
1,gene,Gene,http://www.case.edu/EpilepsyOntology.owl#Gene,http://www.w3.org/2002/07/owl#Class,,EpilepsyOntology.owl#Gene
2,gene,Gene,http://www.co-ode.org/ontologies/galen#Gene,http://www.w3.org/2002/07/owl#Class,,galen#Gene
3,gene,Gene,http://www.semanticweb.org/rjyy/ontologies/201...,http://www.w3.org/2002/07/owl#Class,,ESSO#Gene
4,gene,Gene,http://purl.org/skeletome/bonedysplasia#Gene,http://www.w3.org/2002/07/owl#Class,A functional unit of heredity which occupies a...,bonedysplasia#Gene
5,gene,gene,http://purl.org/biotop/biotop.owl#Gene,http://www.w3.org/2002/07/owl#Class,,biotop.owl#Gene
6,gene,Gene,http://www.biopax.org/release/biopax-level3.ow...,http://www.w3.org/2002/07/owl#Class,Definition: A continuant that encodes informat...,biopax-level3.owl#Gene
7,gene,Gene,http://www.imgt.org/download/IMGT-ONTOLOGY/IMG...,http://www.w3.org/2002/07/owl#Class,"The ""Gene"" concept allows to classify a unit o...",IMGT-ONTOLOGY-v1-0-3.owl#Gene
8,gene,gene,http://www.imgt.org/download/IMGT-ONTOLOGY/IMG...,http://www.w3.org/2002/07/owl#Class,Identifies a gDNA sequence unit that can be po...,IMGT-ONTOLOGY-v1-0-3.owl#gene
9,gene,Gene,http://www.cvrgrid.org/ontologies/Electrophysi...,http://www.w3.org/2002/07/owl#Class,,Electrophysiology#Gene


## 8. Inspect Sample Records and Fields
Fetch full details for a few genomics-related classes and display definitions, synonyms, and relationships.

In [11]:
from urllib.parse import quote


def get_class_details(class_iri: str) -> Dict[str, Any]:
    encoded = quote(class_iri, safe="")
    return api_get_auth(f"/ontologies/{TCO_ACRONYM}/classes/{encoded}")


sample_ids = normalized_results["@id"].dropna().head(5).tolist()

sample_details: List[Dict[str, Any]] = []
for class_iri in sample_ids:
    detail = get_class_details(class_iri)
    sample_details.append({
        "prefLabel": detail.get("prefLabel") or detail.get("label"),
        "@id": detail.get("@id"),
        "definition": normalize_text_list(detail.get("definition")),
        "synonym": normalize_text_list(detail.get("synonym")),
        "parents": normalize_text_list(detail.get("links", {}).get("parents")),
        "children": normalize_text_list(detail.get("links", {}).get("children")),
        "properties": normalize_text_list(detail.get("properties")),
    })

pd.DataFrame(sample_details)

HTTPError: 404 Client Error: Not Found for url: https://data.bioontology.org/ontologies/None/classes/http%3A%2F%2Fwww.projecthalo.com%2Faura%23Gene?apikey=98d19152-8c21-4a0c-bd50-c09b46543947